# Stage 9 — Debit spread live screener

New: `src/debit_spread_screener.py` — parallel to `daily_screener.py`, not
a modification of it. That one checks a stock's currently-open Supertrend
trend AGE; this checks for a currently-OPEN debit-spread TRADE (a signal
that fired recently and hasn't yet resolved) - the actionable entry
itself, with live stop/target and candles left before the time-exit
forces a close.

Same two-speed discipline as the existing screener: deep full-universe
refresh (notebook 08, ~monthly) keeps the ranking honest; THIS notebook
does a quick, short-lookback refresh of only the shortlist, right before
checking - never point this at the full universe.

In [14]:
import sys
sys.path.append("..")

import pandas as pd

ranking = pd.read_excel("../exports/RANKING_debit_spread.xlsx")

# Keep only outlier-reliance-clean stocks, take the top 20
clean_candidates = ranking[
    ranking["outlier_reliance_flag"].isna() | (ranking["outlier_reliance_flag"] == "")
]
top_20 = clean_candidates.head(20)

# Always include NIFTY and BANKNIFTY, same flag check
must_include = ranking[ranking["symbol"].isin(["NIFTY", "BANKNIFTY"])]
must_include_clean = must_include[must_include["outlier_reliance_flag"].isna() | (must_include["outlier_reliance_flag"] == "")]

final_candidates = pd.concat([top_20, must_include_clean]).drop_duplicates(subset="symbol").reset_index(drop=True)
final_candidates[["symbol", "debit_spread_score", "win_rate_pct", "avg_r_achieved", "median_r_achieved"]]

,symbol,debit_spread_score,win_rate_pct,avg_r_achieved,median_r_achieved
0,POWERINDIA,98.9,65.2,0.56,1.50
1,TIINDIA,95.0,54.5,0.28,0.61
2,CDSL,94.5,57.4,0.41,1.46
3,INFY,93.4,56.0,0.34,0.82
4,GAIL,92.8,52.4,0.27,0.50
5,PFC,92.5,54.1,0.36,0.95
6,SUZLON,92.4,54.9,0.33,0.64
7,ADANIENSOL,91.8,52.9,0.29,0.28
8,ANGELONE,91.8,53.9,0.33,0.74
9,BHEL,90.7,55.9,0.30,0.34


In [15]:
from datetime import datetime, timedelta
from src.auth import get_kite
from src.instruments import build_nse_token_lookup, get_index_token
from src.debit_spread_scale_pipeline import process_instrument

kite = get_kite()
token_lookup = build_nse_token_lookup(kite)

symbols_to_refresh = final_candidates["symbol"].tolist()
index_symbols = {"NIFTY": ("NIFTY 50", "NSE"), "BANKNIFTY": ("NIFTY BANK", "NSE")}

# Quick lookback (not the 2023-01-01 deep window from notebook 08) - just
# enough for EMA/ADX warm-up (~45 days) plus room for a still-open trade.
from_date = datetime.now() - timedelta(days=120)

for i, symbol in enumerate(symbols_to_refresh, start=1):
    try:
        if symbol in index_symbols:
            tradingsymbol, exchange = index_symbols[symbol]
            token = get_index_token(kite, tradingsymbol, exchange)
        else:
            token = token_lookup.get(symbol)

        if token is None:
            print(f"[{i}/{len(symbols_to_refresh)}] {symbol} — no token found, skipping")
            continue

        process_instrument(symbol, token, kite, from_date=from_date, to_date=datetime.now())
        print(f"[{i}/{len(symbols_to_refresh)}] {symbol} — refreshed")
    except Exception as e:
        print(f"[{i}/{len(symbols_to_refresh)}] {symbol} — FAILED: {e}")

Fetching 2026-04-06 to 2026-08-04...
Saved 574 candles to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/data/raw/POWERINDIA_60minute.csv
Saved 4 trades to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/debit_spread/POWERINDIA_trades.csv
[1/20] POWERINDIA — refreshed
Fetching 2026-04-06 to 2026-08-04...
Saved 574 candles to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/data/raw/TIINDIA_60minute.csv
Saved 4 trades to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/debit_spread/TIINDIA_trades.csv
[2/20] TIINDIA — refreshed
Fetching 2026-04-06 to 2026-08-04...
Saved 574 candles to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/data/raw/CDSL_60minute.csv
Saved 5 trades to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Fil

In [16]:
from src.debit_spread_screener import screen_shortlist

active = screen_shortlist(symbols_to_refresh)
print(f"{len(active)} of {len(symbols_to_refresh)} shortlisted instruments have an active trade right now")
active

1 of 20 shortlisted instruments have an active trade right now


,symbol,direction,entry_date,entry_price,stop_price,target_price,candles_held,candles_left_before_time_exit,current_pct_move,current_r
0,KPITTECH,UP,2026-08-03 11:15:00+05:30,612.55,597.46,635.19,8,6,2.46,1.0
